In [18]:
import sys
import os
import pandas as pd
from datasets import Dataset

sys.path.append(os.path.abspath(".."))
from utils.preprocess import load_and_preprocess, format_input

In [19]:
from transformers import BioGptTokenizer, BioGptForCausalLM
import torch

print(torch.cuda.is_available())
torch.cuda.empty_cache()

True


In [20]:
tokenizer = BioGptTokenizer.from_pretrained("microsoft/BioGPT")
model = BioGptForCausalLM.from_pretrained("microsoft/BioGPT")

tokenizer.padding_side = "left"

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

In [21]:
def truncate_context(context, max_context_tokens=300):
    if isinstance(context, list):
        context = " ".join(context)
    return tokenizer.decode(
        tokenizer(context, add_special_tokens=False)["input_ids"][:max_context_tokens],
        skip_special_tokens=True
    )

In [22]:
MAX_LEN = 512
ANSWER_RESERVE = 16  # leave room for yes/no/maybe or short answer

def tokenize_pubmedqa(example):
    question = example["question"]
    context = example["context"]
    if isinstance(context, list):
        context = " ".join(context)

    target = example.get("final_decision", example.get("long_answer", ""))

    prompt_prefix = f"Question: {question}\nContext: "
    prompt_suffix = "\nAnswer:"
    target_text = f" {target}"

    prefix_ids = tokenizer(prompt_prefix, add_special_tokens=False)["input_ids"]
    context_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    suffix_ids = tokenizer(prompt_suffix, add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target_text, add_special_tokens=False)["input_ids"]

    max_context_len = MAX_LEN - len(prefix_ids) - len(suffix_ids) - len(target_ids)
    max_context_len = max(0, max_context_len)

    context_ids = context_ids[:max_context_len]

    input_ids = prefix_ids + context_ids + suffix_ids + target_ids
    attention_mask = [1] * len(input_ids)

    labels = (
        [-100] * (len(prefix_ids) + len(context_ids) + len(suffix_ids))
        + target_ids
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [23]:
def preprocess(row):
    # question = row["question"]
    # context = row["context"]
    # if isinstance(context, list):
    #     context = " ".join(context)

    # target_text = row.get("final_decision", row.get("answer", ""))
    # context = truncate_context(context)

    # prompt = f"Question: {question}\nContext: {context}\nAnswer:"
    # target = f" {target_text}"

    # prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    # target_ids = tokenizer(target, add_special_tokens=False)["input_ids"]

    # return {
    #     "input_ids": prompt_ids + target_ids,
    #     "attention_mask": [1] * (len(prompt_ids) + len(target_ids)),
    #     "labels": [-100] * len(prompt_ids) + target_ids,
    # }
    return tokenize_pubmedqa(row)

In [24]:
def format_input(row, test=False):
    context = row['context']
    
    # keep last few sentences
    sentences = context.split('.')
    context = '.'.join(sentences[-3:])
    if not test:
        return f"""source: question: {row['question']} context: {row['context']}
target:  the answer to the question given the context is {row['label']}"""
    else:
        return f"""source: question: {row['question']} context: {row['context']}
target:  the answer to the question given the context is"""

In [25]:
# Load data using the predefined function
train_df = load_and_preprocess('../data/pqal_fold0/train_set.json', False)
test_df = load_and_preprocess('../data/pqal_fold0/dev_set.json', False)

In [26]:
train_df['preprocess'] = train_df.apply(preprocess, axis=1)
train_df['preprocess']

0      {'input_ids': [4790, 20925, 20, 8667, 3776, 22...
1      {'input_ids': [4790, 20925, 20, 10552, 1422, 1...
2      {'input_ids': [4790, 20925, 20, 212, 121, 3583...
3      {'input_ids': [4790, 20925, 20, 10552, 6, 952,...
4      {'input_ids': [4790, 20925, 20, 20650, 27, 407...
                             ...                        
445    {'input_ids': [4790, 20925, 20, 8667, 8961, 15...
446    {'input_ids': [4790, 20925, 20, 8667, 1544, 20...
447    {'input_ids': [4790, 20925, 20, 8667, 32, 881,...
448    {'input_ids': [4790, 20925, 20, 35535, 815, 23...
449    {'input_ids': [4790, 20925, 20, 2882, 982, 101...
Name: preprocess, Length: 450, dtype: object

In [27]:
processed_df = pd.DataFrame(train_df["preprocess"].tolist())
processed_df

,input_ids,attention_mask,labels
0,"[4790, 20925, 20, 8667, 3776, 22101, 33392, 88...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
1,"[4790, 20925, 20, 10552, 1422, 103, 1394, 324,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
2,"[4790, 20925, 20, 212, 121, 35832, 33925, 121,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
3,"[4790, 20925, 20, 10552, 6, 952, 5, 6, 49, 600...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
4,"[4790, 20925, 20, 20650, 27, 407, 1008, 20, 14...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
...,...,...,...
445,"[4790, 20925, 20, 8667, 8961, 1513, 868, 6, 35...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
446,"[4790, 20925, 20, 8667, 1544, 2098, 16, 6, 485...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
447,"[4790, 20925, 20, 8667, 32, 881, 981, 3689, 10...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
448,"[4790, 20925, 20, 35535, 815, 23381, 13562, 10...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."


In [28]:
dataset = Dataset.from_pandas(processed_df[["input_ids", "attention_mask", "labels"]])
dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 450
})

In [29]:
train_df['label'].value_counts(normalize=True)


label
yes      0.553333
no       0.337778
maybe    0.108889
Name: proportion, dtype: float64

In [30]:
# Formatting text to feed input for LLM fine-tuning
# train_df['formatted'] = train_df.apply(format_input, axis=1)
# train_df['formatted'][0]

In [31]:
# Formatting text to feed input for LLM fine-tuning
# test_df['formatted'] = test_df.apply(format_input, axis=1, args=(True,))
# test_df['formatted'][0]

In [32]:
# !pip install transformers
# !pip install sacremoses
# !pip install torch
# !pip install "transformers[torch]"
# !pip install peft

# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [17]:
train_texts = train_df['formatted'].tolist()

# Encoding the text using tokenizer
train_encodings = tokenizer(
    train_texts,
    padding=True, # used padding and truncation to maintain same length of token
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

train_encodings.keys()
tokenizer.decode(train_encodings['input_ids'][0])

KeyError: 'formatted'

In [ ]:
test_texts = test_df['formatted'].tolist()

# Encoding the text using tokenizer
test_encodings = tokenizer(
    test_texts,
    padding=True, # used padding and truncation to maintain same length of token
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

tokenizer.decode(test_encodings['input_ids'][0])


"<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad></s>source: question: Is cytokeratin immunoreactivity useful in the diagnosis of short-segment Barrett's oesophagus in Korea? context: Cytokeratin 7 / 20 staining has been reported to be helpful in diagnosing Barrett's oesophagus and gastric intestinal metaplasia. However, 

In [ ]:
# Adding labels to the text
labels = train_encodings['input_ids'].clone()
labels

tensor([[    1,     1,     1,  ...,  1544,    21, 22700],
        [    1,     1,     1,  ...,  1544,    21, 22700],
        [    1,     1,     1,  ...,  1544,    21, 22700],
        ...,
        [    1,     1,     1,  ...,  1544,    21,   102],
        [    1,     1,     1,  ...,  1544,    21, 22700],
        [    1,     1,     1,  ...,  1544,    21, 22700]])

In [ ]:
# Getting the tokenized value for the word "Answer:"
answer_ids = tokenizer("target:  the answer to the question given the context is", add_special_tokens=False).input_ids
answer_ids

[552, 20, 6, 9412, 13, 6, 2969, 690, 6, 1544, 21]

In [ ]:
for i in range(len(labels)):
    input_id = train_encodings["input_ids"][i]

    for j in range(len(input_id)):
        if (input_id[j:j+len(answer_ids)].tolist() == answer_ids):
            answer_start_index = j
            break
    
    # masking everything except the answer
    # this helps the LLM to learn only answer
    labels[i][:answer_start_index] = -100

In [ ]:
train_encodings["labels"] = labels

In [ ]:
tok = tokenizer.decode(train_encodings["labels"])
split = [r.split("target: the answer to the question given the context is")[-1].strip() for r in tok]
split

['yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'maybe',
 'maybe',
 'maybe',
 'maybe',
 'maybe',
 'maybe',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'maybe',
 'maybe',
 'maybe',
 'maybe',
 'maybe',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 '<unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><u

In [ ]:

train_dataset = Dataset.from_dict(train_encodings)
test_dataset = Dataset.from_dict(test_encodings)

In [ ]:
train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 450
})

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=16,
    warmup_steps=100,

    num_train_epochs=10,
    learning_rate=5e-6,

    logging_steps=50,

    # eval_strategy="steps",
    # eval_steps=500,

    save_steps=500,
    save_total_limit=2,

    fp16=True,

    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    # eval_dataset=test_dataset
)

In [ ]:
trainer.train()

c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(


Step,Training Loss
50,8.026308
100,7.476935
150,6.348044
200,5.289115
250,4.649555


TrainOutput(global_step=290, training_loss=6.09464111328125, metrics={'train_runtime': 731.3694, 'train_samples_per_second': 6.153, 'train_steps_per_second': 0.397, 'total_flos': 4190024761344000.0, 'train_loss': 6.09464111328125, 'epoch': 10.0})

In [ ]:
allowed_words = ["yes", "no", "maybe"]

allowed_token_ids = [
    tokenizer.encode(word, add_special_tokens=False)[0]
    for word in allowed_words
]

In [ ]:
from transformers import LogitsProcessor, LogitsProcessorList

class RestrictTokensProcessor(LogitsProcessor):
    def __init__(self, allowed_token_ids):
        self.allowed_token_ids = allowed_token_ids

    def __call__(self, input_ids, scores):
        mask = torch.full_like(scores, float("-inf"))
        mask[:, self.allowed_token_ids] = 0
        return scores + mask

In [ ]:
logits_processor = LogitsProcessorList([
    RestrictTokensProcessor(allowed_token_ids)
])

In [ ]:
outputs = model.generate(
    **test_encodings.to(model.device),
    max_new_tokens=3,
    # logits_processor=logits_processor,
    do_sample=False
)

In [ ]:
results = tokenizer.batch_decode(outputs, skip_special_tokens=True)

answers = [r.split("target: the answer to the question given the context is")[-1].strip() for r in results]
answers

['"Can cytokeratin',
 'that extended aortic',
 ': Is D',
 '"Does tranexamic',
 ': Should asc',
 '"Do we',
 ': "Does',
 'that the eleph',
 ': Can medical',
 ': Estimation of',
 'not the patient',
 '"Is it',
 ': "Does',
 'that the patient',
 '"Is it',
 'that the changes',
 'that scrotal incision',
 '"Screening /',
 'that the patient',
 '"Does automatic',
 ': "Is',
 'that the h',
 '"Should early',
 'that LA is',
 ': Can a',
 ': "Does',
 'that SLT is',
 'that the factors',
 '"Does the',
 ': Is routine',
 '"Can mass',
 'that the laparoscopic',
 'that the association',
 'that family meetings',
 'that meropenem is',
 '"Can you',
 '"Do you',
 'that the benefit',
 'to know the',
 ': "If',
 '"Does the',
 'that the short',
 '"Does the',
 '"Does your',
 ': "Urine',
 '"Does accompanying',
 'that the follow',
 ': Can APC',
 '" Would',
 ": 'How"]